# Gödel 1931：从符号到数的三个 TOY 实验

目标：亲手检查编码、替换和有限证明验证。不是原形式系统的复刻，不构造自指句，不证明不完备性。先尝试预测每个单元的输出，再运行它。

依赖：Python 3.10+；将本 notebook 与 godel_toy.py 放在同一目录。

In [1]:
from godel_toy import (
    Add, Eq, Nat, Var, ProofStep, bounded_search, check_proof,
    component, concatenate, decode, encode, encode_proof,
    evaluate, formula_code, formula_tokens, proof_relation, substitute_numeral,
)
print("TOY 模块已加载；后面的三个小语言/系统有明确边界。")


TOY 模块已加载；后面的三个小语言/系统有明确边界。


## 1. 序列编码：为什么指数要加一？

这里采用 product(pᵢ^(aᵢ+1))，允许 aᵢ=0。它不同于原文直接以正符号代码作指数的编码。空序列用 1 表示。请先算 [0,2,1] 的代码。

In [2]:
sequence = (0, 2, 1)
n = encode(sequence)
assert n == 1350 and decode(n) == sequence
assert encode(()) == 1 and encode((0,)) == 2
print("代码 =", n, "；解码 =", decode(n))
print("第二个分量（下标1）=", component(n, 1))
print("拼接代码 =", concatenate(encode((0, 2)), encode((1,))))


代码 = 1350 ；解码 = (0, 2, 1)
第二个分量（下标1）= 2
拼接代码 = 1350


中间缺少素因子不是合法的零元素。试问：10=2×5 能代表这个约定下的序列吗？

In [3]:
try:
    decode(10)
except ValueError as error:
    print("按预期拒绝：", error)
else:
    raise AssertionError("含孔的编码应被拒绝")


按预期拒绝： prime gap: a later factor exists after a missing position


## 2. 把变量 x 换成数字项 2

我们显式限制为无量词 AST，没有绑定变量或捕获问题。这是局部教学替换，不是原文完整的 Sb。对公式求值与形式证明是不同任务。

In [4]:
f = Eq(Add(Var("x"), Nat(1)), Nat(3))
g = substitute_numeral(f, "x", 2)
assert g == Eq(Add(Nat(2), Nat(1)), Nat(3))
assert evaluate(g) is True
print("x+1=3 的 tokens：", formula_tokens(f))
print("2+1=3 的 tokens：", formula_tokens(g))
print("编码：", formula_code(f), "→", formula_code(g))
print("自然数解释下为真：", evaluate(g))


x+1=3 的 tokens： (3, 2, 1, 0, 0, 1, 0, 3)
2+1=3 的 tokens： (3, 2, 0, 2, 0, 1, 0, 3)
编码： 311361212962800 → 3051339887035440
自然数解释下为真： True


## 3. 给定证明可以逐行检查

另一个独立的有限系统：公理 A；规则 A→B、B→C、C→D。命题代码依次为 0、1、2、3。证明行存储（命题代码、理由、前提行号）；前提引用从 1 开始且必须指向更早行。这个系统不是实验二的算术语言。

In [5]:
proof = (ProofStep(0, 0, 0), ProofStep(1, 1, 1), ProofStep(2, 2, 2))
p = encode_proof(proof)
assert p == 12324817146749777670
assert check_proof(proof, 2) and proof_relation(p, 2)
bad = (ProofStep(0, 0, 0), ProofStep(2, 2, 1))
assert not check_proof(bad, 2)
print("有效证明 [A,B,C]：p =", p)
print("Proof(p,2) =", proof_relation(p, 2))
print("错误跳步 [A,C] =", check_proof(bad, 2))


有效证明 [A,B,C]：p = 12324817146749777670
Proof(p,2) = True
错误跳步 [A,C] = False


## 4. 有界搜索失败意味着什么？

先预测：C 最少需要几行？比较两个上界。有限上界内没找到，不能直接推出不可证明。

In [6]:
short = bounded_search(2, 2)
longer = bounded_search(2, 3)
assert short.proof is None
assert longer.proof is not None and check_proof(longer.proof, 2)
for result in (short, longer):
    print(f"至多 {result.max_lines} 行：找到={result.proof is not None}；检查={result.candidates_checked}")
print("2 行内的失败并不意味着 C 不可证明。")


至多 2 行：找到=False；检查=3
至多 3 行：找到=True；检查=9
2 行内的失败并不意味着 C 不可证明。


## 5. 运行整个测试集

包括小序列穷举、空序列、缺位编码、上下界、非法输入、替换语义以及证明检查的反例。通过测试不等于证明关于所有自然数的数学结论。

In [7]:
import subprocess
import sys
result = subprocess.run([sys.executable, "-m", "unittest", "-v"], capture_output=True, text=True)
if result.returncode:
    print(result.stdout + result.stderr)
    raise AssertionError("测试失败")
print("\n".join(result.stderr.strip().splitlines()[-4:]))


----------------------------------------------------------------------
Ran 45 tests in 0.007s

OK


## 读完后应能分清

- 代码数字与它代表的符号对象。
- 自然数解释下为真与某个形式系统中可证明。
- 检查一个给定证明与寻找任意长度的证明。
- 有界搜索未找到与元数学证明不可证明。

本系统的有限原子闭包可以另外计算，不能拿它冒充含充分算术的不完备理论。还缺少可表示性、对角化、自指句以及一致性/ω-一致性论证。详见 README.zh-CN.md。

验证说明：此环境未安装 Jupyter 内核；保存输出由随附 stdlib runner 真实顺序执行纯 Python 单元所得，不声称完成 Jupyter 前端/内核测试。